# Econ 521 — Lab Problem Set 1
## Potential Outcomes, Selection Bias, and Randomized Experiments


**Weight:** This problem set is part of the lab component (25% of the course grade). Work may be discussed with classmates, but the code and the write-up you submit must be your own.

**What to submit.** Upload to Canvas:
- This `.ipynb` notebook, completed, that reproduces every number when run top to bottom on a fresh kernel. If it does not run, it is not graded.
- Discussion and interpretation should be written in the markdown cells provided.

**Grading.**

| Component | Weight |
|---|---|
| Part I — Simulation (potential outcomes, SDO decomposition, bias vs. noise) | 35% |
| Part II — Replication: main treatment effects | 10% |
| Part III — Randomization check: session assignments | 25% |
| Part IV — Randomization check: price randomization (Table A2) | 25% |
| Presentation (tables labeled, prose in complete sentences, code runs) | 5% |


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.random.seed(20260911)         # fixed seed — everyone should get the same numbers
rng = np.random.default_rng(20260911)

# Parts II-IV use the authors' original replication data, distributed alongside
# this notebook: 0422analysis.dta (survey + network file, n = 4,902) and
# 0422price.dta (the price-randomization subsample, n = 433). Keep these files
# in the same folder as the notebook.

---
# Part I — Simulation: Potential Outcomes and Selection Bias (35%)

## Background

Recall the Rubin Causal Model from lecture. For each unit $i$:

- $Y_i(1)$: potential outcome under treatment
- $Y_i(0)$: potential outcome under control
- $D_i \in \{0, 1\}$: treatment indicator
- Individual treatment effect: $\delta_i = Y_i(1) - Y_i(0)$
- Realized outcome (switching equation): $Y_i = D_i \cdot Y_i(1) + (1 - D_i) \cdot Y_i(0)$

The **fundamental problem of causal inference** is that we only ever observe one potential outcome per unit. The other one does not exist in the data.

When we compare the average realized outcome of the treated to the average realized outcome of the untreated, we get the **Simple Difference in Outcomes (SDO)**:

$$\text{SDO} = E[Y \mid D=1] - E[Y \mid D=0]$$

From lecture, we know this decomposes as:

$$\text{SDO} = \underbrace{E[Y(1) - Y(0) \mid D=1]}_{\text{ATT}} + \underbrace{E[Y(0) \mid D=1] - E[Y(0) \mid D=0]}_{\text{selection bias}}$$

Randomization eliminates the selection bias term because, under independence $(Y(1), Y(0)) \perp\!\!\!\perp D$, the untreated potential outcome $Y(0)$ has the same distribution in both groups.

## Your data generating process

You are simulating a job training program. The outcome $Y$ is annual earnings in dollars. There are $n = 10{,}000$ individuals.

**Step 1.** Draw a latent "ability" variable:

$$A_i \sim \text{Normal}(\mu = 50, \, \sigma = 10)$$

**Step 2.** Construct potential outcomes:

$$Y_i(0) = 20{,}000 + 500 \cdot A_i + \varepsilon_i, \qquad \varepsilon_i \sim \text{Normal}(0, \, 3000)$$

$$Y_i(1) = Y_i(0) + 1000 + 30 \cdot A_i + \nu_i, \qquad \nu_i \sim \text{Normal}(0, \, 1000)$$

So the individual treatment effect is $\delta_i = 1000 + 30 \cdot A_i + \nu_i$. Treatment effects are heterogeneous: higher-ability individuals benefit more from the program.

**Step 3.** The **Perfect Doctor** assignment rule (sorting on gains):

$$D_i = \mathbf{1}[\delta_i > 0]$$

Because $\delta_i = 1000 + 30 A_i + \nu_i$ is almost always positive (the mean is about 2,500), this rule will assign nearly everyone to treatment. To make the exercise more interesting, use instead:

$$D_i^{\text{perfect}} = \mathbf{1}[\delta_i > \text{median}(\delta)]$$

This assigns the top half of gainers to treatment.

**Step 4.** The **Random Doctor** (coin flip):

$$D_i^{\text{random}} \sim \text{Bernoulli}(0.5)$$

Use the seed `np.random.seed(20260911)` so everyone gets identical draws.

### Question 1.1 — Build the DGP (10 points)

In the cell below, generate the data following the equations above. Create a DataFrame with columns: `A`, `Y0`, `Y1`, `delta`, `D_perfect`, `D_random`.

In [ ]:
# ---- Question 1.1 ------------------------------------------------------------

# ------------------------------------------------------------------------------

### Question 1.2 — Decompose the SDO (10 points)

For **each** assignment rule (Perfect Doctor and Random Doctor):

1. Compute the realized outcome $Y_i$ using the switching equation.
2. Compute the **SDO** = mean of $Y$ among treated minus mean of $Y$ among untreated.
3. Compute the **ATT** = $E[\delta_i \mid D_i = 1]$.
4. Compute the **selection bias** = $E[Y(0) \mid D=1] - E[Y(0) \mid D=0]$.
5. Verify numerically that $\text{SDO} = \text{ATT} + \text{selection bias}$.

Present results in a table. Then answer in the markdown cell below:

- Is the SDO larger or smaller than the ATE under each rule? Why?
- Sign and interpret the selection bias term for the Perfect Doctor. What does it say about who enrolls in the training program?

In [ ]:
# ---- Question 1.2 ------------------------------------------------------------

# ------------------------------------------------------------------------------

**Your interpretation (Question 1.2):**

*(Write here. Discuss the sign and magnitude of selection bias under each rule. Why does the Random Doctor produce an SDO close to the ATE while the Perfect Doctor does not?)*

### Question 1.3 — Bias does not shrink with sample size (10 points)

A common instinct is that bias goes away if you collect more data. Test this.

For each $n$ in `[250, 1_000, 10_000, 100_000]`, run **200 replications** of the following: redraw the full DGP (same equations as above), apply the Perfect Doctor rule, and record the SDO. Then report:

- The **mean SDO** across the 200 replications
- The **standard deviation of the SDO** across replications
- The **mean gap** = mean(SDO) − mean(ATE)

Present this as a table. Then answer: which of these three quantities shrinks as $n$ grows, and which does not? In two sentences, explain why a researcher with a very large observational dataset should not be reassured by the size of the sample alone.

In [ ]:
# ---- Question 1.3 ------------------------------------------------------------

# ------------------------------------------------------------------------------

**Your interpretation (Question 1.3):**

*(Write here. Which column shrinks? Which does not? What does this imply for a researcher with millions of observations but no experiment?)*

### Question 1.4 — Partial sorting (5 points)

Real assignment mechanisms are never perfectly informed. Modify the rule so that with probability $p$ the individual is assigned by the Perfect Doctor rule and with probability $1 - p$ by a coin flip.

For $p$ on a grid from 0 to 1 (in steps of 0.05), compute the **mean gap** (SDO − ATE) over 200 replications with $n = 5{,}000$. Plot the gap against $p$.

What does $p = 0$ correspond to? What does $p = 1$ correspond to? Describe the shape.

In [ ]:
# ---- Question 1.4 ------------------------------------------------------------

# ------------------------------------------------------------------------------

**Your interpretation (Question 1.4):**

*(Write here. What does $p = 0$ mean? What does $p = 1$ mean? What is the shape of the relationship?)*

---
# Part II — Replication: Treatment Effects on Insurance Take-up (10%)

## Background

Cai, de Janvry & Sadoulet (2015), "Social Networks and the Decision to Insure," *American Economic Journal: Applied Economics* 7(2): 81–108.

**Setting.** In rural China, rice farmers were offered a new weather insurance product by the People's Insurance Company of China (PICC). The experiment included 185 villages with 5,332 households. Two treatments were cross-randomized:

1. **Intensive financial education** (`intensive`): households were randomly assigned to either a simple 20-minute session that only introduced the insurance contract, or an intensive 45-minute session that also provided financial education about how insurance works and its expected benefits.

2. **Default option** (`default`): in some villages, the default was set to "buy" (the farmer had to opt out), while in others the default was "not buy" (the farmer had to opt in).

The outcome is `takeup_survey`: whether the household purchased the insurance (0/1).

**Data.** Use `0422analysis.dta`, built from the authors' original AEJ: Applied replication files (the household survey merged with the social-network variables). It contains all 4,902 surveyed households across 47 villages and 173 address groups — the actual estimation sample behind the paper's tables, not a third-party teaching extract.

### Question 2.1 — Summary statistics and main effects (10 points)

1. Load the data. Report $n$, the number of villages, and the number of address groups.
2. Produce a summary statistics table (mean, sd, min, max) for: `takeup_survey`, `intensive`, `default`, `age`, `male`, `literacy`, `agpop`, `ricearea_2010`, `disaster_prob`, `risk_averse`.
3. Estimate the effect of each treatment on `takeup_survey` by OLS, first separately and then jointly. Cluster standard errors at the village level. Report in a table.
4. Interpret: which treatment has a large, significant effect on take-up? Which does not? What is the magnitude in percentage points relative to the control-group mean?


In [ ]:
# ---- Question 2.1 ------------------------------------------------------------
d = pd.read_stata("0422analysis.dta")
# ------------------------------------------------------------------------------

In [ ]:
# ---- Question 2.1, treatment effects ----------------------------------------

# ------------------------------------------------------------------------------

**Your interpretation (Question 2.1):**

*(Write here. Which treatment has the larger effect? How large is each effect in percentage points, and relative to the control-group mean? Can you rule out that either treatment has no effect, or is the study simply unable to detect one of them?)*

---
# Part III — Randomization Check: Session Assignments (25%)

## Background

Table A1 in Cai et al. (2015) checks whether the within-village session randomization succeeded. The idea is simple: if households were truly randomly assigned to different sessions, then observable characteristics should be balanced across session groups. Any large, systematic difference would suggest that the randomization failed or that differential attrition occurred.

In the original paper, households were assigned to one of four sessions: first-round simple (T1), first-round intensive (T2), second-round simple (T3), and second-round intensive (T4). The table presents the mean and standard deviation of each covariate by session, with an F-test p-value for the null that all four session means are equal.

`0422analysis.dta` contains the actual round indicator (`delay`: 0 = first round, 1 = second round) used to build these groups in the paper, so you can construct the *exact* four sessions rather than an approximation:

| `delay` | `intensive` | Session |
|---|---|---|
| 0 | 0 | First round, simple |
| 0 | 1 | First round, intensive |
| 1 | 0 | Second round, simple |
| 1 | 1 | Second round, intensive |

### Question 3.1 — Replicate Table A1 (15 points)

For each of the following covariates: `age`, `male`, `agpop`, `ricearea_2010`, `disaster_prob`, `risk_averse`, `literacy`:

1. Compute the mean and standard deviation **by session group** (four columns, in the order above).
2. Run a one-way ANOVA (F-test) across the four groups and report the F-statistic and p-value.
3. Present this as a single formatted table, with variables as rows and session groups as columns, plus an F-stat and p-value column.
4. Report the number of observations per session group at the bottom of the table, and compare it with the paper's reported group sizes (1,079 / 1,096 / 1,587 / 1,570).


In [ ]:
# ---- Question 3.1 ------------------------------------------------------------

# ------------------------------------------------------------------------------

### Question 3.2 — Interpret the balance table (10 points)

Answer each of the following in the markdown cell below:

1. Are any of the F-test p-values below 0.05? If so, which variable, and is the difference economically meaningful (i.e., large relative to the variable's standard deviation)?

2. You are testing 7 covariates simultaneously. If all are truly balanced, what is the probability that at least one test rejects at the 5% level by chance? Show the calculation.

3. Suppose `ricearea_2010` showed a statistically significant imbalance. Explain what you would do next — would you discard the experiment, add controls, or something else? Justify your answer in terms of the SDO decomposition from Part I.

4. Why do we present standard deviations alongside means in a balance table? What information would be lost if we only reported means and p-values?

**Your answers (Question 3.2):**

*(Write here.)*

---
# Part IV — Randomization Check: Price Randomization (25%)

## Background

Table A2 in Cai et al. (2015) checks the validity of a second randomization: the **price** offered to households in "type II" villages, where the insurance premium itself was randomly assigned across a range of values. Unlike Table A1, the approach here is a regression, not a group comparison: each baseline covariate is regressed on `price` and `price²`, **with address fixed effects**, and the two price coefficients are tested jointly against zero. A rejection would mean the assigned price is predictable from household characteristics — evidence against valid randomization.

**Data.** Use `0422price.dta`, the actual price-randomization subsample from the authors' replication files (433 households across 12 addresses, close to the paper's reported 431). It contains `price`, the assigned premium, along with `age`, `male`, `agpop`, `literacy`, and `earlyarea` (rice production area), and `address`, the fixed-effect group.

This folder also has `analysis.do` (the authors' Stata code) and `analysis.log` (the actual recorded output of running it). Before you write your own regression, it is worth locating the Table A2 block in both files and checking whether the code does what the published table claims — and whether the numbers it actually produces look like the ones in the table above.

### Question 4.1 — Replicate Table A2 (15 points)

For each covariate in `[age, male, agpop, literacy, earlyarea]`:

1. Regress the covariate on `price`, `price²`, and address fixed effects, clustering standard errors by `address`:

$$X_{ia} = \alpha_a + \beta_1 \cdot \texttt{price}_i + \beta_2 \cdot \texttt{price}_i^2 + \varepsilon_{ia}$$

2. Report $\hat\beta_1$, $\hat\beta_2$, and their standard errors.
3. Test $H_0: \beta_1 = \beta_2 = 0$ jointly (an F-test) and report the p-value.
4. Present the results in a table similar to Table A2, with the covariate names as rows.
5. As a robustness check, re-estimate **without** the address fixed effects (pooled) and comment on whether any conclusion changes.


In [ ]:
# ---- Question 4.1 ------------------------------------------------------------
price_data = pd.read_stata("0422price.dta")

# ------------------------------------------------------------------------------

### Question 4.2 — Compare with the published Table A2 (10 points)

The paper reports (price coef., price-squared coef., joint p-value):

| Covariate | Price coef. | Price² coef. | Joint p-value |
|---|---|---|---|
| Gender of household head | 0.007 | 0.001 | 0.4374 |
| Age | -0.331 | 0.096 | 0.2317 |
| Household size | 0.105 | -0.013 | 0.8798 |
| Literate | 0.0113 | -0.001 | 0.9845 |
| Area of rice production | 1.085 | -0.123 | 0.7783 |

1. Place your replication next to these published numbers and compute the differences.
2. Does dropping the address fixed effects (your Q4.1 robustness check) change any conclusion at the 5% level?
3. Why is the *joint* test of $\beta_1$ and $\beta_2$ the right test here, rather than looking at either coefficient by itself? (Hint: think about why price enters as a quadratic.)
4. Interpret: does the price randomization look valid? What would a rejection have implied?


In [ ]:
# ---- Question 4.2 ------------------------------------------------------------

# ------------------------------------------------------------------------------

**Your interpretation (Question 4.2):**

*(Write here. Does the replication match the paper's Table A2 in sign and magnitude? Do any joint tests approach significance? What would a rejection have implied for the credibility of the price randomization? Also note anything you noticed when you compared `analysis.do`'s actual Table A2 code to the published table's description — does the code do exactly what the table claims?)*


---
## A note on the data

Parts II–IV use the authors' original replication files for Cai, de Janvry & Sadoulet (2015) — `0422analysis.dta` for Parts II–III and `0422price.dta` for Part IV — rather than a third-party teaching extract, so your numbers should line up closely with the paper's published tables.

**Reference.**
Cai, Jing, Alain de Janvry, and Elisabeth Sadoulet (2015), "Social Networks and the Decision to Insure," *American Economic Journal: Applied Economics* 7(2): 81–108.
